# **IMPORT LIBRARIES**

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from statsmodels.tsa.arima.model import ARIMA

import warnings
warnings.filterwarnings("ignore")

# **LOAD DATASET**

In [ ]:
df = pd.read_csv('/content/tesla_deliveries_dataset_2015_2025.csv')

print(df.head())

In [ ]:
print(df.shape)

print(df.info())

print(df.describe())

print(df.columns)

In [ ]:
print(df.isnull().sum())

sns.heatmap(df.isnull(), cbar=False)
plt.title("Missing Values")
plt.show()

In [ ]:
print("Duplicates Before:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates After:", df.duplicated().sum())

In [ ]:
print(df.describe().T)

# **DISTIBUTION PLOT**

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Estimated_Deliveries'], kde=True)
plt.title("Distribution of Deliveries")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Model', y='Estimated_Deliveries', data=df)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x='Region', y='Estimated_Deliveries', data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(10,6))
sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm'
)
plt.show()

# **FEATURE ENGINEERING**

In [ ]:
df['Date'] = pd.to_datetime(
    df['Year'].astype(str) + '-' +
    df['Month'].astype(str) + '-01'
)

df['Quarter'] = df['Date'].dt.quarter

# **LABEL ENCODING**

In [ ]:
le_region = LabelEncoder()
le_model = LabelEncoder()
le_source = LabelEncoder()

df['Region'] = le_region.fit_transform(df['Region'])
df['Model'] = le_model.fit_transform(df['Model'])
df['Source_Type'] = le_source.fit_transform(df['Source_Type'])

# **FEATURE SELECTION**

In [ ]:
target = 'Estimated_Deliveries'

In [ ]:
X = df.drop(
    columns=[
        'Estimated_Deliveries',
        'Date'
    ]
)

y = df[target]

# **TRAIN TEST SPLIT**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

# **LINEAR REGRESSION**

In [ ]:
lr = LinearRegression()

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

print("Linear Regression")

print("MAE:",
      mean_absolute_error(y_test, lr_pred))

print("RMSE:",
      np.sqrt(mean_squared_error(y_test, lr_pred)))

print("R2:",
      r2_score(y_test, lr_pred))

# **DECISION TREE**

In [ ]:
dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("Decision Tree")

print("MAE:",
      mean_absolute_error(y_test, dt_pred))

print("RMSE:",
      np.sqrt(mean_squared_error(y_test, dt_pred)))

print("R2:",
      r2_score(y_test, dt_pred))

# **RANDOM FOREST**

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest")

print("MAE:",
      mean_absolute_error(y_test, rf_pred))

print("RMSE:",
      np.sqrt(mean_squared_error(y_test, rf_pred)))

print("R2:",
      r2_score(y_test, rf_pred))

# **Hyperparameter Tuning**

In [ ]:
param_grid = {
    'n_estimators':[100,200],
    'max_depth':[5,10,15,None],
    'min_samples_split':[2,5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid.fit(X_train,y_train)

print("Best Parameters:")
print(grid.best_params_)

In [ ]:
best_rf = grid.best_estimator_

best_pred = best_rf.predict(X_test)

**Feature Importance Analysis**

In [ ]:
importance = pd.DataFrame({
    'Feature':X.columns,
    'Importance':best_rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=importance
)

plt.title("Feature Importance")
plt.show()

# **Time Series Forecasting (ARIMA)**

In [ ]:
ts = df.groupby('Date')[
    'Estimated_Deliveries'
].sum()

In [ ]:
model = ARIMA(
    ts,
    order=(5,1,0)
)

arima_model = model.fit()

print(arima_model.summary())

In [ ]:
forecast = arima_model.forecast(
    steps=12
)

print(forecast)

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(ts, label='Actual')

future_dates = pd.date_range(
    start=ts.index[-1],
    periods=13,
    freq='M'
)[1:]

plt.plot(
    future_dates,
    forecast,
    label='Forecast'
)

plt.legend()
plt.show()

# **Model Comparison**

In [ ]:
results = pd.DataFrame({
    'Model':[
        'Linear Regression',
        'Decision Tree',
        'Random Forest'
    ],
    'R2 Score':[
        r2_score(y_test, lr_pred),
        r2_score(y_test, dt_pred),
        r2_score(y_test, rf_pred)
    ],
    'RMSE':[
        np.sqrt(mean_squared_error(y_test, lr_pred)),
        np.sqrt(mean_squared_error(y_test, dt_pred)),
        np.sqrt(mean_squared_error(y_test, rf_pred))
    ]
})

print(results)

In [ ]:
sns.barplot(
    x='Model',
    y='R2 Score',
    data=results
)

plt.title("Model Comparison")
plt.show()

# **Actual vs Predicted Analysis**

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    rf_pred
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.title("Actual vs Predicted")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, rf_pred)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.title("Actual vs Predicted (Random Forest)")
plt.show()